In [3]:
# import packages
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import shap
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
seed = 2724

### Import data

In [ ]:
DF_PATH = "mod04_data/sample.csv"
df = pd.read_csv(DF_PATH)

### Separate data by independent (X) and dependent (y) variables

In [ ]:
X = df[["income", "education_years", "zipcode_score"]]
y = df["target"]

### Split the data into a _training_ set (to build a model) and _test_ set (to validate a model)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed
)

### Build a model on the training set

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=seed
)
model.fit(X_train, y_train)

### Use SHAP to explain the model on test data

In [ ]:
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

This will allow us to see which variables are most important to predicting the outcome.

In [ ]:
shap.plots.bar(shap_values)

### Import the `group` variable, which was **not** used in training this model.

In [ ]:
X_test_with_group = X_test.copy()
X_test_with_group["group"] = df.loc[X_test.index, "group"]

### Look at the difference in SHAP values between the two groups across the variables used in the model.

In [ ]:
shap_df = pd.DataFrame(shap_values.values, columns=X_test.columns)
shap_df["group"] = X_test_with_group["group"].values

shap_df.groupby("group").mean()

NameError: name 'shap_values' is not defined

### Let's put `group` and `zipcode_score` in the same plot

In [ ]:
def plot_shap(var):
    # Extract SHAP values for the feature
    shap_var = shap_values[:, var].values

    # Plot the values of each group using different colors
    plt.figure()
    plt.scatter(
        X_test[var],
        shap_var,
        c=X_test_with_group["group"]
    )
    plt.xlabel(var)
    plt.ylabel(f"SHAP value for var")
    plt.title("Proxy feature impact by group")
    plt.show()

plot_shap("zipcode_score")

# Discussion Questions

### What is a _SHAP_ (or Shapley) value? 

A SHAP value is a measure that quantifies the contribution of each feature to a model's prediction for a specific instance. For each feature, the SHAP value represents how much that feature pushed the prediction away from the average prediction. A positive SHAP value means the feature increased the prediction, while a negative SHAP value means it decreased the prediction.

### Suppose you built this model and then it is peer reviewed by another entity. If the reviewer asks whether you used the variable `group` in your model, what would your answer be?

No, the group variable was not used. The only features used for training were income, education_years, and zipcode_score.

### If the reviewer asks whether the outcome of your model is correlated with `group`, what would your answer be?

Yes, the outcome of the model is correlated with group, even though it was not used as a feature. The groupby analysis shows different mean SHAP values between groups.

### Construct a "proxy feature impact by group" plot for `income`. How is this plot different from the one for `zipcode_score`?

In [ ]:
plot_shap("income")

Unlike the zipcode_score plot where the two groups are clearly separated along the x-axis, the income plot shows both groups mixed together across all income values.

### If, instead, you were the **reviewer**, what other questions might you ask the person who built this model? Give at least two.


What would happen if you removed zipcode_score from the model? Would model accuracy decrease? Is education_years also correlated with group?